In [2]:
import pandas as pd
import statsmodels api as sm
import statsmodels.formula.api as smf

# Sample dataset
data = pd.DataFrame({
    'Accidents': [0,1,3,2,4,1,0,2],
    'Age': [25,30,45,50,40,35,28,60],
    'Experience': [1,5,20,25,15,10,3,30]
})

# Poisson regression
model = smf.glm(formula="Accidents ~ Age + Experience", 
                data=data, 
                family=sm.families.Poisson()).fit()

print(model.summary())


SyntaxError: invalid syntax (1480254926.py, line 2)

In [1]:
import time
import sys

def type_lyric(line, char_delay=0.06):
    for char in line:
        print(char, end="", flush=True)
        time.sleep(char_delay)
    print()

def play_reel():
    lyrics = [
        "Secrets I have held in my heart",
        "Are harder to hide than I thought",
        "Maybe I just wanna be yours",
        "I wanna be yours, I wanna be yours",
        "Wanna be yours",
        "Wanna be yours",
        "Wanna be yours"
    ]

    delays = [1.8, 1.8, 2.2, 2.5, 1.8, 1.8, 2.0]

    print("\n🎧 Now Playing: \"I Wanna Be Yours\" - Arctic Monkeys (Python Reel) 🎧\n")
    time.sleep(1.5)

    for i, line in enumerate(lyrics):
        type_lyric(line)
        time.sleep(delays[i])

play_reel()


🎧 Now Playing: "I Wanna Be Yours" - Arctic Monkeys (Python Reel) 🎧

Secrets I have held in my heart
Are harder to hide than I thought
Maybe I just wanna be yours
I wanna be yours, I wanna be yours
Wanna be yours
Wanna be yours
Wanna be yours


In [3]:
"""
lyrics_player.py
Plays song.mp3 and shows synced lyrics from lyrics.lrc in a small tkinter window.

Requirements:
    pip install pygame
Run:
    python lyrics_player.py
Place song.mp3 and lyrics.lrc in the same directory as this script.
"""

import pygame
import time
import threading
import tkinter as tk
from tkinter import ttk

# ---------- Config ----------
AUDIO_FILE = "song.mp3"    # put your audio file here
LYRICS_FILE = "lyrics.lrc" # lrc-style file with [mm:ss.xx] timestamps
REFRESH_INTERVAL = 0.1     # seconds between GUI updates
# ----------------------------

def parse_lrc(path):
    """
    Parse a simple LRC file and return a sorted list of (time_seconds, text).
    Supports timestamps like [mm:ss.xx] or [mm:ss].
    Ignores lines without timestamps.
    """
    entries = []
    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            # A line can have multiple timestamps for same text: [00:10.00][00:20.00]text
            parts = []
            rest = line
            while rest.startswith('['):
                closing = rest.find(']')
                if closing == -1:
                    break
                stamp = rest[1:closing]
                parts.append(stamp)
                rest = rest[closing+1:]
            text = rest.strip()
            for stamp in parts:
                try:
                    # stamp format mm:ss.xx or mm:ss
                    if ':' not in stamp:
                        continue
                    mm, ss = stamp.split(':', 1)
                    seconds = int(mm) * 60 + float(ss)
                    entries.append((seconds, text))
                except Exception:
                    # ignore bad stamps
                    continue
    entries.sort(key=lambda x: x[0])
    return entries

class LyricsPlayer:
    def __init__(self, audio_file, lyrics):
        self.audio_file = audio_file
        self.lyrics = lyrics  # list of (time, text)
        self.current_index = 0
        self.play_start_time = None
        self.is_playing = False

        pygame.mixer.init()
        pygame.mixer.music.load(self.audio_file)

        # GUI
        self.root = tk.Tk()
        self.root.title("Lyrics Player")
        self.root.geometry("700x300")
        self.root.resizable(True, True)

        self.frame = ttk.Frame(self.root, padding=10)
        self.frame.pack(fill=tk.BOTH, expand=True)

        # Scrolling text area
        self.text_area = tk.Text(self.frame, wrap='word', state='disabled', font=("Helvetica", 16))
        self.text_area.pack(fill=tk.BOTH, expand=True, side=tk.LEFT)

        self.scrollbar = ttk.Scrollbar(self.frame, command=self.text_area.yview)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.text_area['yscrollcommand'] = self.scrollbar.set

        # Populate text area with all lyrics
        self._populate_lyrics()

        # Controls
        controls = ttk.Frame(self.root, padding=(10,5))
        controls.pack(fill=tk.X)
        self.play_btn = ttk.Button(controls, text="Play", command=self.play_pause)
        self.play_btn.pack(side=tk.LEFT)
        self.stop_btn = ttk.Button(controls, text="Stop", command=self.stop)
        self.stop_btn.pack(side=tk.LEFT, padx=(6,0))

        # Start update loop
        self._update_loop()

        # Ensure pygame stops on close
        self.root.protocol("WM_DELETE_WINDOW", self._on_close)

    def _populate_lyrics(self):
        self.text_area.configure(state='normal')
        self.text_area.delete('1.0', tk.END)
        for idx, (_, line) in enumerate(self.lyrics):
            # insert each line and tag it with an index
            self.text_area.insert(tk.END, line + '\n')
            tag_name = f"line_{idx}"
            start = f"{idx+1}.0"
            end = f"{idx+1}.end"
            self.text_area.tag_add(tag_name, start, end)
        self.text_area.configure(state='disabled')

    def play_pause(self):
        if not self.is_playing:
            # start or resume playback
            # Use pygame to play; set play_start_time to current time minus already played pos
            pygame.mixer.music.play(start=0)  # note: start param behavior varies per file type
            self.play_start_time = time.time()
            self.is_playing = True
            self.play_btn.config(text="Pause")
        else:
            # pause
            pygame.mixer.music.pause()
            # store elapsed time so we can resume
            self.paused_at = time.time() - self.play_start_time
            self.is_playing = False
            self.play_btn.config(text="Play")

    def stop(self):
        pygame.mixer.music.stop()
        self.is_playing = False
        self.play_start_time = None
        self.current_index = 0
        self._highlight_line(-1)
        self.play_btn.config(text="Play")

    def _get_elapsed(self):
        """
        Return elapsed seconds since playback started.
        If paused or not started, returns 0 or paused_at.
        """
        if not self.is_playing:
            return getattr(self, 'paused_at', 0.0)
        return time.time() - self.play_start_time

    def _update_loop(self):
        # Called periodically to sync lyrics
        elapsed = self._get_elapsed()
        idx = self.current_index

        # find which lyric should currently be shown
        # advance index while next lyric time <= elapsed
        while idx + 1 < len(self.lyrics) and self.lyrics[idx + 1][0] <= elapsed:
            idx += 1
        # if current index is still greater than 0 but current lyric time > elapsed, step back
        while idx > 0 and self.lyrics[idx][0] > elapsed:
            idx -= 1

        if idx != self.current_index:
            self.current_index = idx
            self._highlight_line(idx)

        # check for end of song
        if self.is_playing:
            # if music finished, pygame.mixer.music.get_busy() becomes False
            if not pygame.mixer.music.get_busy():
                # playback finished
                self.is_playing = False
                self.play_btn.config(text="Play")

        # schedule next update
        self.root.after(int(REFRESH_INTERVAL * 1000), self._update_loop)

    def _highlight_line(self, idx):
        # remove previous highlight
        self.text_area.configure(state='normal')
        self.text_area.tag_remove("highlight", "1.0", tk.END)
        # normal style for all lines
        for i in range(len(self.lyrics)):
            tag_name = f"line_{i}"
            self.text_area.tag_config(tag_name, foreground="black")
        if 0 <= idx < len(self.lyrics):
            tag_name = f"line_{idx}"
            # configure highlight - bold and slightly larger via font (basic approach)
            self.text_area.tag_config(tag_name, foreground="blue")
            # scroll so the highlighted line is visible
            line_index = f"{idx+1}.0"
            self.text_area.see(line_index)
        self.text_area.configure(state='disabled')

    def _on_close(self):
        try:
            pygame.mixer.music.stop()
            pygame.mixer.quit()
        except Exception:
            pass
        self.root.destroy()

    def run(self):
        self.root.mainloop()

def main():
    try:
        lyrics = parse_lrc(LYRICS_FILE)
        if not lyrics:
            print("No lyrics found in", LYRICS_FILE)
            print("Make sure the file exists and uses [mm:ss.xx] timestamps.")
            return
    except FileNotFoundError:
        print(f"Could not find {LYRICS_FILE}. Create it with lrc timestamps.")
        return

    try:
        player = LyricsPlayer(AUDIO_FILE, lyrics)
        player.run()
    except Exception as e:
        print("Error:", e)

if __name__ == "__main__":
    main()


pygame 2.6.1 (SDL 2.28.4, Python 3.10.10)
Hello from the pygame community. https://www.pygame.org/contribute.html
Could not find lyrics.lrc. Create it with lrc timestamps.


In [3]:
import pandas as pd
import statsmodels.formula.api as smf

# ✅ Step 1: Create some sample data (replace with your real data)
data = {
    "Weight": [50, 52, 54, 60, 62, 65, 70, 72],
    "Age":    [1, 2, 3, 1, 2, 3, 1, 2],
    "Farm":   ["A", "A", "A", "B", "B", "B", "C", "C"]
}

df = pd.DataFrame(data)  # ✅ now df is defined

# ✅ Step 2: Fit a random regression model
model = smf.mixedlm("Weight ~ Age", data=df, groups=df["Farm"], re_formula="~Age")
result = model.fit()
print(result.summary())


IndentationError: unexpected indent (mixed_linear_model.py, line 2200)

In [4]:
import pygame
import time

# Initialize pygame mixer
pygame.mixer.init()

# Load your vibe sound (replace with your file name)
pygame.mixer.music.load("vibe_sound.mp3")

# Play the music
pygame.mixer.music.play()

print("🎶 Playing your vibe sound... Relax and enjoy!")

# Keep the program running while the music plays
while pygame.mixer.music.get_busy():
    time.sleep(1)


error: No file 'vibe_sound.mp3' found in working directory 'd:\datapreprocess'.

In [4]:
import pandas as pd

df = pd.read_csv(r"D:\dataprocess\weatherHistory.csv")
df.head()
df.describe()

,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars)
count,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.000000,96453.0,96453.000000
mean,11.932678,10.855029,0.734899,10.810640,187.509232,10.347325,0.0,1003.235956
std,9.551546,10.696847,0.195473,6.913571,107.383428,4.192123,0.0,116.969906
min,-21.822222,-27.716667,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,4.688889,2.311111,0.600000,5.828200,116.000000,8.339800,0.0,1011.900000
50%,12.000000,12.000000,0.780000,9.965900,180.000000,10.046400,0.0,1016.450000
75%,18.838889,18.838889,0.890000,14.135800,290.000000,14.812000,0.0,1021.090000
max,39.905556,39.344444,1.000000,63.852600,359.000000,16.100000,0.0,1046.380000


In [5]:
"""
Python Banner Generator
-----------------------
Create beautiful ASCII text banners with colors and styles.

Requirements:
    pip install pyfiglet colorama
"""

import os
import pyfiglet
from colorama import Fore, Style, init

# Initialize colorama
init(autoreset=True)

# Available colors
COLORS = {
    "red": Fore.RED,
    "green": Fore.GREEN,
    "yellow": Fore.YELLOW,
    "blue": Fore.BLUE,
    "magenta": Fore.MAGENTA,
    "cyan": Fore.CYAN,
    "white": Fore.WHITE,
}

# Available fonts (you can see all with: pyfiglet.FigletFont.getFonts())
FONTS = [
    "slant", "3-d", "5lineoblique", "standard", "banner3-D", "digital",
    "isometric1", "block", "bubble", "doom"
]


def clear_screen():
    """Clears the terminal for a fresh look."""
    os.system('cls' if os.name == 'nt' else 'clear')


def print_title():
    """Prints the program title."""
    title = pyfiglet.figlet_format("Banner Generator", font="slant")
    print(Fore.CYAN + title)
    print(Fore.YELLOW + "=" * 60)
    print(Fore.GREEN + "Create beautiful text banners easily in Python!")
    print(Fore.YELLOW + "=" * 60 + "\n")


def choose_color():
    """Asks user to choose a color."""
    print(Fore.YELLOW + "Available colors:")
    for color in COLORS.keys():
        print(" -", color)
    color_choice = input(Fore.CYAN + "\nEnter color name: ").strip().lower()
    return COLORS.get(color_choice, Fore.WHITE)


def choose_font():
    """Asks user to choose a font."""
    print(Fore.YELLOW + "\nAvailable fonts:")
    for i, font in enumerate(FONTS, 1):
        print(f" {i}. {font}")
    try:
        choice = int(input(Fore.CYAN + "\nChoose font number: "))
        if 1 <= choice <= len(FONTS):
            return FONTS[choice - 1]
    except ValueError:
        pass
    print(Fore.RED + "Invalid choice! Using default font 'standard'.")
    return "standard"


def create_banner(text, font, color):
    """Creates and displays a banner."""
    banner = pyfiglet.figlet_format(text, font=font)
    print(color + banner + Style.RESET_ALL)


def main():
    clear_screen()
    print_title()

    while True:
        text = input(Fore.GREEN + "\nEnter text for banner (or 'exit' to quit): ").strip()
        if text.lower() == "exit":
            print(Fore.MAGENTA + "\nThanks for using the Banner Generator! 👋")
            break

        color = choose_color()
        font = choose_font()

        clear_screen()
        print_title()
        print(Fore.WHITE + "Your generated banner:\n")
        create_banner(text, font, color)

        again = input(Fore.YELLOW + "\nDo you want to create another banner? (y/n): ").lower()
        if again != "y":
            print(Fore.CYAN + "\nGoodbye! Keep coding in Python 💻")
            break


if __name__ == "__main__":
    main()


    ____                             
   / __ )____ _____  ____  ___  _____
  / __  / __ `/ __ \/ __ \/ _ \/ ___/
 / /_/ / /_/ / / / / / / /  __/ /    
/_____/\__,_/_/ /_/_/ /_/\___/_/     
                                     
   ______                           __            
  / ____/__  ____  ___  _________ _/ /_____  _____
 / / __/ _ \/ __ \/ _ \/ ___/ __ `/ __/ __ \/ ___/
/ /_/ /  __/ / / /  __/ /  / /_/ / /_/ /_/ / /    
\____/\___/_/ /_/\___/_/   \__,_/\__/\____/_/     
                                                  

Create beautiful text banners easily in Python!

Available colors:
 - red
 - green
 - yellow
 - blue
 - magenta
 - cyan
 - white

Available fonts:
 1. slant
 2. 3-d
 3. 5lineoblique
 4. standard
 5. banner3-D
 6. digital
 7. isometric1
 8. block
 9. bubble
 10. doom
Invalid choice! Using default font 'standard'.
    ____                             
   / __ )____ _____  ____  ___  _____
  / __  / __ `/ __ \/ __ \/ _ \/ ___/
 / /_/ / /_/ / / / / / / /  __/ /    

In [6]:
"""
Rolls Royce Banner Generator 🚗
--------------------------------
A colorful banner display with a luxury Rolls Royce theme.
Make sure to install these first:
    pip install pyfiglet colorama time
"""

import os
import time
import pyfiglet
from colorama import Fore, Back, Style, init

# Initialize colorama
init(autoreset=True)

# Function to clear screen
def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

# Loading animation
def loading_animation():
    print(Fore.YELLOW + "\nStarting Rolls Royce Banner Engine...\n")
    for i in range(20):
        print(Fore.CYAN + "🚗" * (i % 5 + 1), end="\r")
        time.sleep(0.15)
    print(Fore.GREEN + "✅ Engine Ready!\n")
    time.sleep(0.5)

# Rolls Royce logo banner
def rolls_royce_logo():
    logo = pyfiglet.figlet_format("Rolls Royce", font="slant")
    print(Fore.MAGENTA + Style.BRIGHT + logo)

# Luxury tagline banner
def luxury_banner(text):
    font_style = "standard"
    banner = pyfiglet.figlet_format(text, font=font_style)
    print(Fore.CYAN + banner + Style.RESET_ALL)

# Car ASCII Art
def car_ascii():
    car = r"""
          ______
         //  ||\ \
   ____//||\ \__
   )  _          _    \
   |/ \/ \|
__\/\/_____
    """
    print(Fore.LIGHTWHITE_EX + car)
    print(Fore.YELLOW + "🏁 The Spirit of Ecstasy 🏁")
    print(Fore.LIGHTBLUE_EX + "Luxury | Power | Perfection\n")

# Fancy rainbow effect text
def rainbow_text(text):
    colors = [Fore.RED, Fore.YELLOW, Fore.GREEN, Fore.CYAN, Fore.MAGENTA, Fore.BLUE]
    for i, char in enumerate(text):
        print(colors[i % len(colors)] + char, end="")
        time.sleep(0.05)
    print(Style.RESET_ALL)

# Main display
def main():
    clear_screen()
    loading_animation()
    rolls_royce_logo()
    car_ascii()

    rainbow_text("Welcome to the Rolls Royce Python Banner Experience 🚘")
    print()
    time.sleep(1)

    while True:
        text = input(Fore.LIGHTGREEN_EX + "\nEnter your custom banner text (or 'exit' to quit): ").strip()
        if text.lower() == "exit":
            print(Fore.MAGENTA + "\n👋 Exiting Rolls Royce Banner Generator... Safe Drive!")
            break

        clear_screen()
        rolls_royce_logo()
        car_ascii()
        luxury_banner(text)
        rainbow_text("✨ Driven by Elegance ✨")

        again = input(Fore.YELLOW + "\nDo you want to create another banner? (y/n): ").lower()
        if again != "y":
            print(Fore.CYAN + "\nGoodbye! Keep shining like Rolls Royce 💫")
            break

if __name__ == "__main__":
    main()



Starting Rolls Royce Banner Engine...



✅ Engine Ready!

    ____        ____        ____                      
   / __ \____  / / /____   / __ \____  __  __________ 
  / /_/ / __ \/ / / ___/  / /_/ / __ \/ / / / ___/ _ \
 / _, _/ /_/ / / (__  )  / _, _/ /_/ / /_/ / /__/  __/
/_/ |_|\____/_/_/____/  /_/ |_|\____/\__, /\___/\___/ 
                                    /____/            


          ______
         //  ||\ \
   ____//||\ \__
   )  _          _    \
   |/ \/ \|
__\/\/_____
    
🏁 The Spirit of Ecstasy 🏁
Luxury | Power | Perfection

Welcome to the Rolls Royce Python Banner Experience 🚘

    ____        ____        ____                      
   / __ \____  / / /____   / __ \____  __  __________ 
  / /_/ / __ \/ / / ___/  / /_/ / __ \/ / / / ___/ _ \
 / _, _/ /_/ / / (__  )  / _, _/ /_/ / /_/ / /__/  __/
/_/ |_|\____/_/_/____/  /_/ |_|\____/\__, /\___/\___/ 
                                    /____/            


          ______
         //  ||\ \
   ____//||\ \__
   )  _          _    \
   |/ \/ \|
__\/\/_____
  

In [7]:
import speech_recognition as sr
import pyttsx3
import datetime
import random
import re  # For simple math parsing
import sys

# Initialize recognizer and TTS engine
recognizer = sr.Recognizer()
tts_engine = pyttsx3.init()

# Set TTS properties (voice speed and volume)
tts_engine.setProperty('rate', 150)  # Speed of speech
tts_engine.setProperty('volume', 0.9)  # Volume (0.0 to 1.0)

# List of microphones (use default)
microphone = sr.Microphone()

# Adjust for ambient noise
with microphone as source:
    recognizer.adjust_for_ambient_noise(source, duration=1)

def speak(text):
    """Convert text to speech."""
    print(f"Assistant: {text}")
    tts_engine.say(text)
    tts_engine.runAndWait()

def listen():
    """Listen for voice input and return recognized text."""
    try:
        print("Listening... Speak now!")
        with microphone as source:
            # Listen for audio with timeout and phrase time limit
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
        
        print("Processing...")
        # Recognize speech using Google's free API (requires internet)
        text = recognizer.recognize_google(audio).lower()
        print(f"You said: {text}")
        return text
    except sr.WaitTimeoutError:
        return None
    except sr.UnknownValueError:
        speak("Sorry, I didn't catch that. Could you repeat?")
        return None
    except sr.RequestError as e:
        speak(f"Speech service error: {e}")
        sys.exit(1)

def process_command(command):
    """Simple 'AI' processing: Rule-based intent recognition."""
    if not command:
        return "I didn't hear anything. Let's try again."
    
    command = command.lower().strip()
    
    # Greeting intents
    if any(word in command for word in ['hello', 'hi', 'hey']):
        responses = ["Hello! How can I help you today?", "Hi there! What's up?"]
        return random.choice(responses)
    
    # Time query
    elif 'time' in command or 'clock' in command:
        current_time = datetime.datetime.now().strftime("%I:%M %p")
        return f"The current time is {current_time}."
    
    # Date query
    elif 'date' in command or 'day' in command:
        current_date = datetime.datetime.now().strftime("%B %d, %Y")
        return f"Today is {current_date}."
    
    # Simple math (e.g., "calculate 2 + 2" or "what is 5 * 3")
    elif 'calculate' in command or 'what is' in command:
        # Extract numbers and operator using regex
        math_match = re.search(r'(\d+)\s*([+\-*/])\s*(\d+)', command)
        if math_match:
            num1, op, num2 = math_match.groups()
            num1, num2 = int(num1), int(num2)
            if op == '+':
                result = num1 + num2
            elif op == '-':
                result = num1 - num2
            elif op == '*':
                result = num1 * num2
            elif op == '/':
                if num2 != 0:
                    result = num1 / num2
                else:
                    return "Cannot divide by zero!"
            return f"The result is {result}."
        else:
            return "I couldn't parse the math. Try something like 'calculate 2 + 2'."
    
    # Joke
    elif 'joke' in command or 'funny' in command:
        jokes = [
            "Why don't scientists trust atoms? Because they make up everything!",
            "Why did the scarecrow win an award? He was outstanding in his field!",
            "What do you call fake spaghetti? An impasta!"
        ]
        return random.choice(jokes)
    
    # Exit
    elif any(word in command for word in ['exit', 'bye', 'goodbye', 'quit']):
        return "Goodbye! Have a great day."
    
    # Fallback
    else:
        return "I'm not sure how to help with that yet. Try asking for the time, a joke, or some math!"

def main():
    speak("Hello! I'm your voice assistant. Say something, or 'exit' to quit.")
    
    while True:
        command = listen()
        if command is None:
            continue  # Retry on failure
        
        response = process_command(command)
        speak(response)
        
        # Check for exit in response
        if 'goodbye' in response.lower():
            break

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        speak("Shutting down. Bye!")
        sys.exit(0)


Assistant: Hello! I'm your voice assistant. Say something, or 'exit' to quit.
Listening... Speak now!
Processing...
Assistant: Speech service error: recognition connection failed: [Errno 11001] getaddrinfo failed


SystemExit: 1

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
# Nature Scene Generator 🌿
# Author: Vishwanathan R
# A colorful terminal art of a nature scene using Python

from colorama import Fore, Back, Style, init
import time
import random

# Initialize colorama
init(autoreset=True)

def draw_sky():
    sky = ""
    for i in range(5):
        line = ""
        for j in range(50):
            if random.choice([True, False, False]):
                line += Fore.WHITE + "☁"
            else:
                line += Fore.CYAN + " "
        sky += line + "\n"
    return sky

def draw_sun():
    return Fore.YELLOW + "       ☀☀☀\n" + Fore.YELLOW + "        ☀☀\n"

def draw_trees():
    tree = ""
    for _ in range(5):
        space = " " * random.randint(2, 8)
        tree += (
            space + Fore.GREEN + "  🌲  " + "\n" +
            space + Fore.GREEN + "  🌳  " + "\n"
        )
    return tree

def draw_birds():
    birds = ""
    for i in range(3):
        space = " " * random.randint(5, 40)
        birds += Fore.WHITE + space + "🕊🕊" + "\n"
    return birds

def draw_grass():
    grass = ""
    for _ in range(3):
        line = ""
        for i in range(50):
            line += Fore.GREEN + random.choice(["🌿", "🍀", "🌱"])
        grass += line + "\n"
    return grass

def main():
    print(Fore.BLUE + Style.BRIGHT + "\n🌄  Welcome to Python Nature Scene Generator 🌄\n")
    time.sleep(1)

    print(draw_sky())
    time.sleep(0.5)
    print(draw_sun())
    time.sleep(0.5)
    print(draw_birds())
    time.sleep(0.5)
    print(draw_trees())
    time.sleep(0.5)
    print(draw_grass())
    time.sleep(0.5)

    print(Fore.MAGENTA + "\n🌸 Enjoy the beauty of code and nature! 🌸\n")

if __name__ == "__main__":
    main()


🌄  Welcome to Python Nature Scene Generator 🌄



  ☁☁☁   ☁☁☁☁          ☁☁☁ ☁      ☁        ☁☁☁     
  ☁☁☁ ☁☁☁☁☁☁ ☁  ☁  ☁☁    ☁☁ ☁      ☁☁☁  ☁ ☁☁      
  ☁☁     ☁  ☁   ☁ ☁  ☁☁ ☁☁  ☁ ☁  ☁ ☁      ☁  ☁☁ ☁☁
      ☁☁ ☁☁ ☁☁    ☁    ☁      ☁       ☁☁ ☁ ☁☁☁☁   
     ☁   ☁       ☁    ☁☁  ☁  ☁    ☁☁     ☁   ☁    

       ☀☀☀
        ☀☀

         🕊🕊
                      🕊🕊
                               🕊🕊

         🌲  
         🌳  
          🌲  
          🌳  
       🌲  
       🌳  
        🌲  
        🌳  
          🌲  
          🌳  

🍀🌿🌱🌱🌿🌱🌿🌿🌿🌿🌿🌿🌱🌱🍀🌱🌿🌱🌱🌱🌿🍀🌱🌱🍀🌿🍀🌿🌱🍀🌱🌿🌿🌱🌱🌱🌿🍀🌿🌿🌿🌿🍀🌱🌱🍀🌿🌱🌱🌱
🍀🌿🌿🌱🍀🍀🍀🍀🌿🍀🍀🍀🍀🌱🍀🌿🌱🍀🍀🍀🌱🌿🍀🌿🌱🌱🍀🌿🌱🌿🍀🍀🌱🌿🍀🌿🌱🍀🌿🍀🌱🌿🍀🍀🍀🌿🌱🍀🌿🍀
🍀🌿🌱🌿🌱🌿🍀🌿🍀🌿🍀🌱🍀🍀🌿🍀🌿🌿🌱🍀🌱🍀🌿🌱🌱🌿🌱🌱🍀🌿🍀🌿🌱🍀🌿🌿🍀🍀🍀🌿🌿🍀🍀🌿🍀🌱🌿🍀🌿🌱


🌸 Enjoy the beauty of code and nature! 🌸



In [8]:
import speech_recognition as sr
import pyttsx3
import datetime
import json
import os
import random
import re
import sys
from typing import List, Dict

# Initialize speech components
recognizer = sr.Recognizer()
tts_engine = pyttsx3.init()
tts_engine.setProperty('rate', 150)
tts_engine.setProperty('volume', 0.9)
microphone = sr.Microphone()

# Adjust for ambient noise
with microphone as source:
    recognizer.adjust_for_ambient_noise(source, duration=1)

# File paths for personal data storage
TODOS_FILE = 'todos.json'
FLIGHTS_FILE = 'flights.json'

class Todo:
    """Simple Todo class for personal tasks."""
    def __init__(self):
        self.tasks: List[Dict] = self.load_todos()

    def load_todos(self) -> List[Dict]:
        if os.path.exists(TODOS_FILE):
            with open(TODOS_FILE, 'r') as f:
                return json.load(f)
        return []

    def save_todos(self):
        with open(TODOS_FILE, 'w') as f:
            json.dump(self.tasks, f, indent=4)

    def add_task(self, task_text: str) -> bool:
        if task_text.strip():
            self.tasks.append({'task': task_text.strip(), 'done': False})
            self.save_todos()
            return True
        return False

    def view_tasks(self) -> str:
        if not self.tasks:
            return "No todos yet."
        details = []
        for i, task in enumerate(self.tasks, 1):
            status = "✓ Done" if task['done'] else "○ Pending"
            details.append(f"{i}. {task['task']} [{status}]")
        return "\n".join(details)

    def mark_done(self, index: int) -> bool:
        if 0 <= index < len(self.tasks):
            self.tasks[index]['done'] = True
            self.save_todos()
            return True
        return False

class Flight:
    """Flight class from previous project (simplified for personal use)."""
    def __init__(self, flight_number: str, origin: str, destination: str, capacity: int = 5):
        self.flight_number = flight_number.strip().upper()
        self.origin = origin.strip()
        self.destination = destination.strip()
        self.capacity = capacity
        self.passengers: List[str] = []
        self.departed = False

    def add_passenger(self, passenger_name: str) -> bool:
        if self.departed or len(self.passengers) >= self.capacity:
            return False
        if passenger_name.strip():
            self.passengers.append(passenger_name.strip())
            return True
        return False

    def display_details(self) -> str:
        status = "Departed" if self.departed else f"Active ({len(self.passengers)}/{self.capacity})"
        details = [f"Flight {self.flight_number}: {self.origin} → {self.destination} [{status}]"]
        if self.passengers:
            details.append("Passengers: " + ", ".join(self.passengers))
        return "\n".join(details)

class PersonalFlights:
    """Manages personal flights list."""
    def __init__(self):
        self.flights: List[Flight] = self.load_flights()

    def load_flights(self) -> List[Flight]:
        if os.path.exists(FLIGHTS_FILE):
            with open(FLIGHTS_FILE, 'r') as f:
                data = json.load(f)
                return [Flight(**flight_data) for flight_data in data]  # Reconstruct objects
        return []

    def save_flights(self):
        data = []
        for flight in self.flights:
            data.append({
                'flight_number': flight.flight_number,
                'origin': flight.origin,
                'destination': flight.destination,
                'capacity': flight.capacity,
                'passengers': flight.passengers,
                'departed': flight.departed
            })
        with open(FLIGHTS_FILE, 'w') as f:
            json.dump(data, f, indent=4)

    def create_flight(self, flight_number: str, origin: str, destination: str, capacity: int = 5) -> bool:
        new_flight = Flight(flight_number, origin, destination, capacity)
        self.flights.append(new_flight)
        self.save_flights()
        return True

    def view_flights(self) -> str:
        if not self.flights:
            return "No flights created yet."
        details = []
        for flight in self.flights:
            details.append(flight.display_details())
        return "\n".join(details)

# Global instances for personal use
todos = Todo()
flights = PersonalFlights()

def speak(text: str):
    """Text-to-speech output."""
    print(f"Assistant: {text}")
    tts_engine.say(text)
    tts_engine.runAndWait()

def listen() -> str:
    """Capture and recognize voice input."""
    try:
        print("Listening... (Speak now or wait 5 seconds)")
        with microphone as source:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
        # For offline: Uncomment below and install pocketsphinx
        # text = recognizer.recognize_sphinx(audio).lower()
        text = recognizer.recognize_google(audio).lower()
        print(f"You said: {text}")
        return text
    except sr.WaitTimeoutError:
        return ""
    except sr.UnknownValueError:
        speak("Sorry, I didn't understand. Please repeat.")
        return ""
    except sr.RequestError as e:
        speak(f"Speech error: {e}. Using offline mode next time.")
        return ""

def process_personal_command(command: str) -> str:
    """AI-like processing for personal commands (rule-based for simplicity)."""
    if not command:
        return "I didn't hear you. Try again."

    command = command.lower().strip()

    # Personal Todo Commands
    if 'add todo' in command or 'new task' in command:
        task = command.replace('add todo', '').replace('new task', '').strip()
        if todos.add_task(task):
            return f"Todo added: {task}"
        return "Couldn't add that todo."

    elif 'view todos' in command or 'show tasks' in command:
        details = todos.view_tasks()
        return f"Your todos:\n{details}" if details != "No todos yet." else details

    elif 'mark done' in command:
        # Simple: Assumes first todo for demo; customize for index
        if todos.mark_done(0):
            return "First todo marked as done!"
        return "No todos to mark."

    # Personal Flight Commands
    elif 'create flight' in command:
        # Parse: e.g., "create flight AA123 to Paris from New York"
        parts = command.split('create flight')[1].strip()
        flight_num_match = re.search(r'([A-Z0-9]+)', parts)
        to_match = re.search(r'to\s+([a-z\s]+)', parts)
        from_match = re.search(r'from\s+([a-z\s]+)', parts)
        flight_num = flight_num_match.group(1) if flight_num_match else "PersonalFlight"
        dest = to_match.group(1).strip() if to_match else "Destination"
        origin = from_match.group(1).strip() if from_match else "Origin"
        if flights.create_flight(flight_num, origin, dest):
            return f"Flight {flight_num} created: {origin} to {dest}"
        return "Couldn't create flight."

    elif 'view flights' in command or 'show flights' in command:
        details = flights.view_flights()
        return f"Your flights:\n{details}" if details != "No flights created yet." else details

    # General Personal Commands
    elif 'time' in command:
        current_time = datetime.datetime.now().strftime("%I:%M %p")
        return f"It's {current_time}."

    elif 'date' in command:
        current_date = datetime.datetime.now().strftime("%B %d, %Y")
        return f"Today is {current_date}."

    elif 'calculate' in command or 'math' in command:
        math_match = re.search(r'(\d+)\s*([+\-*/])\s*(\d+)', command)
        if math_match:
            num1, op, num2 = int(math_match.group(1)), math_match.group(2), int(math_match.group(3))
            if op == '+': result = num1 + num2
            elif op == '-': result = num1 - num2
            elif op == '*': result = num1 * num2
            elif op == '/': result = num1 / num2 if num2 != 0 else "Error: Divide by zero"
            return f"{num1} {op} {num2} = {result}"
        return "Couldn't calculate that. Try 'calculate 2 + 2'."

    elif 'joke' in command:
        jokes = ["Why did the computer go to therapy? It had too many bytes of emotional baggage!", "I'm reading a book on anti-gravity—it's impossible to put down!"]
        return random.choice(jokes)

    elif any(word in command for word in ['exit', 'bye', 'stop']):
        return "Shutting down. See you later!"

    else:
        return "For personal use, try: 'add todo [task]', 'view todos', 'create flight [details]', 'time', or 'joke'."

def main():
    speak("Welcome to your personal assistant! Speak a command, like 'add todo buy groceries' or 'what time is it'. Say 'exit' to quit.")
    
    while True:
        command = listen()
        response = process_personal_command(command)
        speak(response)
        
        if 'shutting down' in response.lower() or 'see you later' in response.lower():
            break

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        speak("Goodbye!")
        sys.exit(0)


Assistant: Welcome to your personal assistant! Speak a command, like 'add todo buy groceries' or 'what time is it'. Say 'exit' to quit.
Listening... (Speak now or wait 5 seconds)
Assistant: Speech error: recognition connection failed: [Errno 11001] getaddrinfo failed. Using offline mode next time.
Assistant: I didn't hear you. Try again.
Listening... (Speak now or wait 5 seconds)
Assistant: Speech error: recognition connection failed: [Errno 11001] getaddrinfo failed. Using offline mode next time.
Assistant: I didn't hear you. Try again.
Listening... (Speak now or wait 5 seconds)
Assistant: Speech error: recognition connection failed: [Errno 11001] getaddrinfo failed. Using offline mode next time.
Assistant: I didn't hear you. Try again.
Listening... (Speak now or wait 5 seconds)
Assistant: Speech error: recognition connection failed: [Errno 11001] getaddrinfo failed. Using offline mode next time.
Assistant: I didn't hear you. Try again.
Listening... (Speak now or wait 5 seconds)
Assis

SystemExit: 0

In [9]:
"""
DCGAN for generating car designs (PyTorch).
Place training images in: data/cars/*.jpg
Run: python dcgan_car_generator.py --mode train
Or:   python dcgan_car_generator.py --mode sample
"""

import os
import argparse
import random
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from PIL import Image
import matplotlib.pyplot as plt

# -------------------------
# Config & Hyperparameters
# -------------------------
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", type=str, default="data/cars", help="folder with car images")
parser.add_argument("--out_dir", type=str, default="outputs", help="where to save models and samples")
parser.add_argument("--image_size", type=int, default=64, help="image size (64 recommended for DCGAN)")
parser.add_argument("--batch_size", type=int, default=128)
parser.add_argument("--z_dim", type=int, default=100, help="latent vector size")
parser.add_argument("--g_features", type=int, default=64, help="generator feature maps base")
parser.add_argument("--d_features", type=int, default=64, help="discriminator feature maps base")
parser.add_argument("--lr", type=float, default=2e-4)
parser.add_argument("--beta1", type=float, default=0.5)
parser.add_argument("--epochs", type=int, default=40)
parser.add_argument("--seed", type=int, default=999)
parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
parser.add_argument("--mode", type=str, choices=["train","sample"], default="train")
parser.add_argument("--sample_count", type=int, default=64)
args = parser.parse_args()

os.makedirs(args.out_dir, exist_ok=True)
torch.manual_seed(args.seed)
random.seed(args.seed)

# -------------------------
# Dataset + Transforms
# -------------------------
transform = transforms.Compose([
    transforms.Resize((args.image_size, args.image_size)),
    transforms.CenterCrop(args.image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))  # range [-1,1]
])

dataset = datasets.ImageFolder(root=os.path.dirname(args.data_dir) if os.path.isdir(os.path.dirname(args.data_dir)) else args.data_dir,
                              transform=transform)

# If ImageFolder doesn't work due to root expectations, fallback to custom glob loader:
if len(dataset) == 0:
    # custom simple loader
    class SimpleImageDataset(torch.utils.data.Dataset):
        def _init_(self, folder, transform):
            self.files = sorted(glob(os.path.join(folder, "*")))
            self.transform = transform
        def _len_(self): return len(self.files)
        def _getitem_(self, idx):
            img = Image.open(self.files[idx]).convert("RGB")
            return self.transform(img)
    dataset = SimpleImageDataset(args.data_dir, transform)

dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=True)

# -------------------------
# DCGAN Model Components
# -------------------------
# Weight initialization (DCGAN paper)
def weights_init(m):
    classname = m._class.name_
    if classname.find('Conv') != -1 or classname.find("Linear") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Generator
class Generator(nn.Module):
    def _init_(self, z_dim=100, g_features=64, out_channels=3):
        super()._init_()
        self.net = nn.Sequential(
            # input Z latent vector
            # z -> (g_features*8) x 4 x 4
            nn.ConvTranspose2d(z_dim, g_features*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(g_features*8),
            nn.ReLU(True),

            # state size: (g_features*8) x 4 x 4 -> (g_features*4) x 8 x 8
            nn.ConvTranspose2d(g_features*8, g_features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_features*4),
            nn.ReLU(True),

            # -> (g_features*2) x 16 x 16
            nn.ConvTranspose2d(g_features*4, g_features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_features*2),
            nn.ReLU(True),

            # -> (g_features) x 32 x 32
            nn.ConvTranspose2d(g_features*2, g_features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(g_features),
            nn.ReLU(True),

            # -> 3 x 64 x 64
            nn.ConvTranspose2d(g_features, out_channels, 4, 2, 1, bias=False),
            nn.Tanh()  # output in [-1,1]
        )

    def forward(self, x):
        return self.net(x)

# Discriminator
class Discriminator(nn.Module):
    def _init_(self, d_features=64, in_channels=3):
        super()._init_()
        self.net = nn.Sequential(
            # input 3 x 64 x 64
            nn.Conv2d(in_channels, d_features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features, d_features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_features*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features*2, d_features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_features*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(d_features*4, d_features*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(d_features*8),
            nn.LeakyReLU(0.2, inplace=True),

            # final: 1x1 conv to scalar
            nn.Conv2d(d_features*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()  # probability real/fake
        )

    def forward(self, x):
        return self.net(x).view(-1, 1).squeeze(1)

# -------------------------
# Instantiate models, optimizers, loss
# -------------------------
device = torch.device(args.device)
G = Generator(args.z_dim, args.g_features).to(device)
D = Discriminator(args.d_features).to(device)
G.apply(weights_init)
D.apply(weights_init)

criterion = nn.BCELoss()
optimizerD = torch.optim.Adam(D.parameters(), lr=args.lr, betas=(args.beta1, 0.999))
optimizerG = torch.optim.Adam(G.parameters(), lr=args.lr, betas=(args.beta1, 0.999))

fixed_noise = torch.randn(64, args.z_dim, 1, 1, device=device)  # for monitoring

# -------------------------
# Training loop
# -------------------------
def train():
    real_label = 1.
    fake_label = 0.
    iters = 0
    print("Starting Training Loop on device:", device)
    for epoch in range(args.epochs):
        for i, data in enumerate(tqdm(dataloader)):
            # Get real images
            real_images = data.to(device) if isinstance(data, torch.Tensor) else data[0].to(device)
            b_size = real_images.size(0)

            # ---------------------
            # Update Discriminator: maximize log(D(x)) + log(1 - D(G(z)))
            # ---------------------
            D.zero_grad()
            labels = torch.full((b_size,), real_label, device=device)
            output = D(real_images)
            lossD_real = criterion(output, labels)
            lossD_real.backward()

            noise = torch.randn(b_size, args.z_dim, 1, 1, device=device)
            fake = G(noise)
            labels.fill_(fake_label)
            output = D(fake.detach())
            lossD_fake = criterion(output, labels)
            lossD_fake.backward()
            lossD = lossD_real + lossD_fake
            optimizerD.step()

            # ---------------------
            # Update Generator: maximize log(D(G(z))) <-> minimize BCE(D(G(z)), 1)
            # ---------------------
            G.zero_grad()
            labels.fill_(real_label)  # want generator to trick discriminator
            output = D(fake)
            lossG = criterion(output, labels)
            lossG.backward()
            optimizerG.step()

            if iters % 500 == 0:
                print(f"Epoch [{epoch+1}/{args.epochs}] Iter {iters} LossD: {lossD.item():.4f} LossG: {lossG.item():.4f}")

            iters += 1

        # Save sample images using fixed noise
        with torch.no_grad():
            fake_fixed = G(fixed_noise).detach().cpu()
        utils.save_image((fake_fixed + 1) / 2.0, os.path.join(args.out_dir, f"sample_epoch_{epoch+1}.png"), nrow=8)
        # Save model checkpoints
        torch.save(G.state_dict(), os.path.join(args.out_dir, f"G_epoch_{epoch+1}.pth"))
        torch.save(D.state_dict(), os.path.join(args.out_dir, f"D_epoch_{epoch+1}.pth"))

    print("Training finished.")

# -------------------------
# Sampling (load checkpoint and generate)
# -------------------------
def sample(checkpoint=None, count=64):
    if checkpoint:
        G.load_state_dict(torch.load(checkpoint, map_location=device))
    G.eval()
    z = torch.randn(count, args.z_dim, 1, 1, device=device)
    with torch.no_grad():
        fake = G(z).cpu()
    grid = utils.make_grid((fake + 1) / 2.0, nrow=int(count**0.5), padding=2)
    # Save and display
    out_path = os.path.join(args.out_dir, "sample_output.png")
    utils.save_image(grid, out_path)
    print("Sample saved to", out_path)

    # Display inline (useful if running locally)
    img = transforms.ToPILImage()(grid)
    img.show()

# -------------------------
# Entrypoint
# -------------------------
if _name_ == "_main_":
    if args.mode == "train":
        train()
    else:
        # sample mode: try to load latest checkpoint automatically
        ckpts = sorted(glob(os.path.join(args.out_dir, "G_epoch_*.pth")))
        ckpt = ckpts[-1] if ckpts else None
        if ckpt:
            print("Loading checkpoint:", ckpt)
        sample(checkpoint=ckpt, count=args.sample_count)

usage: ipykernel_launcher.py [-h] [--data_dir DATA_DIR] [--out_dir OUT_DIR]
                             [--image_size IMAGE_SIZE]
                             [--batch_size BATCH_SIZE] [--z_dim Z_DIM]
                             [--g_features G_FEATURES]
                             [--d_features D_FEATURES] [--lr LR]
                             [--beta1 BETA1] [--epochs EPOCHS] [--seed SEED]
                             [--device DEVICE] [--mode {train,sample}]
                             [--sample_count SAMPLE_COUNT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\ADMIN\AppData\Roaming\jupyter\runtime\kernel-v3122b34feb7de7a2855479039f1dd150a27d2535a.json


SystemExit: 2

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
# 🌈 Nature Scene Art in Python (Fully Colorful + Animated)
# Author: Vishwanathan R

from colorama import Fore, Back, Style, init
import os
import time
import random

# Initialize colorama
init(autoreset=True)

# Clear screen function
def clear():
    os.system('cls' if os.name == 'nt' else 'clear')

# Draw the bright blue sky with clouds ☁ and sun ☀
def draw_sky():
    sky = ""
    for _ in range(5):
        line = ""
        for _ in range(60):
            char = random.choice([" ", "☁", " "])
            line += Fore.CYAN + char
        sky += line + "\n"
    return sky

# Draw rainbow 🌈 and shining sun ☀
def draw_rainbow_and_sun():
    rainbow_colors = [Fore.RED, Fore.MAGENTA, Fore.YELLOW, Fore.GREEN, Fore.CYAN, Fore.BLUE]
    rainbow = "".join(color + "🌈" for color in rainbow_colors)
    sun = Fore.YELLOW + "     ☀  ☀  ☀"
    return f"\n{rainbow}\n{sun}\n"

# Draw flying birds 🕊
def draw_birds():
    birds = ""
    for _ in range(2):
        space = " " * random.randint(10, 50)
        birds += Fore.WHITE + space + "🕊🕊" + "\n"
    return birds

# Draw trees 🌳 and flowers 🌸
def draw_trees_and_flowers():
    trees = ""
    for _ in range(4):
        space = " " * random.randint(3, 12)
        trees += (
            space + Fore.GREEN + "  🌳  " + "\n" +
            space + Fore.GREEN + "  🌲  " + "\n" +
            space + Fore.MAGENTA + "  🌺🌸🌷  " + "\n"
        )
    return trees

# Draw river 🌊 and grass 🌿
def draw_river_and_grass():
    river = ""
    for _ in range(2):
        line = ""
        for _ in range(60):
            line += Fore.BLUE + random.choice(["🌊", "💧"])
        river += line + "\n"
    grass = ""
    for _ in range(3):
        line = ""
        for _ in range(60):
            line += random.choice([
                Fore.GREEN + "🌿", 
                Fore.LIGHTGREEN_EX + "🍀",
                Fore.YELLOW + "🌱"
            ])
        grass += line + "\n"
    return river + grass

# Main display
def show_scene():
    clear()
    print(Fore.LIGHTBLUE_EX + Style.BRIGHT + "\n🌄  WELCOME TO COLORFUL PYTHON NATURE WORLD 🌄\n")
    print(draw_sky())
    print(draw_rainbow_and_sun())
    print(draw_birds())
    print(draw_trees_and_flowers())
    print(draw_river_and_grass())
    print(Fore.LIGHTMAGENTA_EX + Style.BRIGHT + "\n🌸 ENJOY THE BEAUTY OF NATURE AND CODE 🌸\n")

# Animate the scene (refresh every second)
def main():
    for i in range(10):  # Loop 10 times to make it move
        show_scene()
        time.sleep(1)

if __name__ == "__main__":
    main()


🌄  WELCOME TO COLORFUL PYTHON NATURE WORLD 🌄

 ☁☁ ☁    ☁   ☁☁    ☁☁        ☁        ☁☁☁  ☁  ☁☁ ☁☁☁☁       
  ☁ ☁   ☁  ☁   ☁ ☁         ☁☁ ☁  ☁ ☁  ☁       ☁   ☁☁  ☁ ☁☁☁☁
 ☁ ☁☁ ☁ ☁ ☁ ☁  ☁  ☁     ☁  ☁    ☁ ☁ ☁ ☁ ☁☁ ☁    ☁    ☁   ☁ ☁
     ☁   ☁   ☁    ☁ ☁ ☁   ☁☁☁ ☁  ☁ ☁☁☁          ☁☁☁☁  ☁☁  ☁ 
 ☁    ☁☁ ☁☁ ☁ ☁       ☁   ☁ ☁         ☁    ☁☁☁ ☁   ☁   ☁    


🌈🌈🌈🌈🌈🌈
     ☀  ☀  ☀

                             🕊🕊
                       🕊🕊

             🌳  
             🌲  
             🌺🌸🌷  
       🌳  
       🌲  
       🌺🌸🌷  
            🌳  
            🌲  
            🌺🌸🌷  
         🌳  
         🌲  
         🌺🌸🌷  

🌊🌊💧🌊💧🌊🌊🌊🌊🌊💧🌊🌊💧💧💧💧🌊💧💧💧🌊🌊💧🌊🌊🌊🌊🌊🌊💧💧💧🌊💧🌊💧💧💧💧🌊🌊💧💧🌊🌊🌊🌊🌊💧💧💧💧💧💧💧💧💧💧💧
🌊🌊🌊🌊💧💧💧💧🌊🌊🌊🌊🌊🌊🌊🌊🌊💧🌊💧🌊🌊💧💧🌊🌊🌊🌊💧🌊💧🌊🌊💧💧🌊💧💧💧🌊💧💧🌊💧🌊🌊💧💧🌊🌊🌊🌊💧🌊💧🌊💧🌊🌊💧
🌱🌱🌿🌱🌱🍀🌱🌱🍀🍀🌿🍀🍀🍀🌿🌿🌿🍀🌱🌱🌿🍀🌱🍀🌿🍀🌱🍀🌱🌿🌿🍀🍀🍀🌱🌱🌿🌱🌿🌱🌱🍀🍀🌱🌱🌱🌿🌿🍀🌱🌱🍀🍀🌿🌿🌱🌱🌱🌱🍀
🌱🌱🌱🌱🌿🌱🌿🍀🌱🌱🌱🌱🍀🌿🌱🌱🌿🍀🌿🌱🌱🌱🍀🌱🌿🌱🌿🌱🌱🌿🍀🌱🌿🌱🌿🌱🌱🌱🍀🌿🌿🍀🌿🍀🌿🌱🍀🌱🌱🍀🌿🌱🌱🍀🌿🌿🌱🍀🌱🌿
🍀🌱🌱🌿🍀🌿🌿🍀🌿🍀🌿🌱🍀🌱🌿🌿🌿🌿🍀🍀🌱🍀🍀🌱🍀🌿🌱🌿🌿🌿🌱🌿🍀🌱🌿🌿🍀🌱🍀🌱🍀🍀🌿🌿🌱🌿🌱🍀🌿🌱🌿🍀🍀🌱🌿🌱🌿🌱🌿🌱


🌸 ENJOY THE BEAUTY OF NATURE AND CODE 🌸


🌄  WELCOME TO COLORFUL PYTHON NATURE 

In [7]:
import speech_recognition as sr
from gtts import gTTS
from langdetect import detect
import datetime
import random
import re
import os
import sys
from playsound import playsound

recognizer = sr.Recognizer()
microphone = sr.Microphone()

# Adjust for noise
with microphone as source:
    recognizer.adjust_for_ambient_noise(source, duration=1)

# ---- Speak Function ----
def speak(text, lang="en"):
    """Convert text to speech and play."""
    print(f"Assistant: {text}")
    tts = gTTS(text=text, lang=lang)
    tts.save("response.mp3")
    playsound("response.mp3")
    os.remove("response.mp3")

# ---- Listen Function ----
def listen():
    """Listen and recognize voice automatically."""
    try:
        print("\n🎧 Listening... Speak now!")
        with microphone as source:
            audio = recognizer.listen(source, timeout=6, phrase_time_limit=6)
        print("Processing voice...")
        text = recognizer.recognize_google(audio, language="ta-IN")  # Default base
        print(f"You said: {text}")
        return text
    except sr.WaitTimeoutError:
        speak("I didn’t hear anything. Please try again.")
        return None
    except sr.UnknownValueError:
        speak("Sorry, I didn’t catch that. Could you repeat?")
        return None
    except sr.RequestError as e:
        speak(f"Speech service error: {e}")
        sys.exit(1)

# ---- Command Processor ----
def process_command(command):
    """Process logic for commands."""
    if not command:
        return "I didn't hear anything.", "en"

    # Detect language dynamically
    try:
        lang_detected = detect(command)
    except:
        lang_detected = "en"

    if lang_detected.startswith("ta"):
        lang = "ta"
    elif lang_detected.startswith("ml"):
        lang = "ml"
    else:
        lang = "en"

    cmd = command.lower().strip()

    # Greetings
    if any(word in cmd for word in ['hello', 'hi', 'hey', 'வணக்கம்', 'ஹை', 'ഹൈ']):
        if lang == "ta":
            return "வணக்கம்! உங்களுக்கு என்ன உதவி தேவை?", lang
        elif lang == "ml":
            return "നമസ്കാരം! എനിക്ക് നിങ്ങളെ എങ്ങനെ സഹായിക്കാം?", lang
        else:
            return "Hello! How can I help you?", lang

    # Time
    elif any(w in cmd for w in ['time', 'நேரம்', 'സമയം']):
        current_time = datetime.datetime.now().strftime("%I:%M %p")
        if lang == "ta":
            return f"இப்போதைய நேரம் {current_time}", lang
        elif lang == "ml":
            return f"ഇപ്പോൾ സമയം {current_time}", lang
        else:
            return f"The current time is {current_time}.", lang

    # Date
    elif any(w in cmd for w in ['date', 'தேதி', 'തീയതി']):
        current_date = datetime.datetime.now().strftime("%B %d, %Y")
        if lang == "ta":
            return f"இன்று {current_date}", lang
        elif lang == "ml":
            return f"ഇന്ന് തീയതി {current_date}", lang
        else:
            return f"Today is {current_date}.", lang

    # Math
    elif any(w in cmd for w in ['calculate', 'எண்ணு', 'കണക്കാക്കൂ']):
        math_match = re.search(r'(\d+)\s*([+\-/])\s(\d+)', cmd)
        if math_match:
            n1, op, n2 = math_match.groups()
            n1, n2 = int(n1), int(n2)
            if op == '+': res = n1 + n2
            elif op == '-': res = n1 - n2
            elif op == '*': res = n1 * n2
            elif op == '/': res = n1 / n2 if n2 != 0 else "undefined"
            if lang == "ta":
                return f"பதில் {res}", lang
            elif lang == "ml":
                return f"ഉത്തരം {res}", lang
            else:
                return f"The result is {res}.", lang
        else:
            return "I couldn’t understand the math problem.", lang

    # Joke
    elif any(w in cmd for w in ['joke', 'காமெடி', 'ചിരി']):
        jokes = [
            ("Why did the computer go to therapy? It had too many bytes!", "en"),
            ("பூனை ஏன் கம்ப்யூட்டர் மேல் அமர்ந்தது? மௌஸ் பிடிக்க!", "ta"),
            ("ഒരു കമ്പ്യൂട്ടർ ഡോക്ടറെ കാണാൻ പോയി. കാരണം അതിന് വൈറസ് പിടിച്ചു!", "ml")
        ]
        return random.choice(jokes)

    # Exit
    elif any(w in cmd for w in ['exit', 'bye', 'goodbye', 'பை', 'விடை', 'വിട']):
        if lang == "ta":
            return "பை! நல்ல நாளாகட்டும்!", lang
        elif lang == "ml":
            return "വിട! നല്ല ദിവസം ആശംസിക്കുന്നു!", lang
        else:
            return "Goodbye! Have a nice day!", lang

    # Default fallback
    else:
        if lang == "ta":
            return "மன்னிக்கவும், இதை எனக்கு புரியவில்லை.", lang
        elif lang == "ml":
            return "ക്ഷമിക്കണം, എനിക്ക് അത് മനസ്സിലായില്ല.", lang
        else:
            return "Sorry, I didn’t understand that.", lang

# ---- Main Function ----
def main():
    speak("Hello! I can understand English, Tamil and Malayalam. Talk to me!", "en")

    while True:
        command = listen()
        if not command:
            continue
        response, lang = process_command(command)
        speak(response, lang)
        if any(x in response for x in ['Goodbye', 'பை', 'വിട']):
            break

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("Exiting program...")


Assistant: Hello! I can understand English, Tamil and Malayalam. Talk to me!

🎧 Listening... Speak now!
Processing voice...
You said: என்னடா பண்ற
Assistant: மன்னிக்கவும், இதை எனக்கு புரியவில்லை.

🎧 Listening... Speak now!
Processing voice...
You said: எதுனா பேசு
Assistant: மன்னிக்கவும், இதை எனக்கு புரியவில்லை.

🎧 Listening... Speak now!
Processing voice...
You said: கேன் யூ டெல் மீ சம்திங்
Assistant: மன்னிக்கவும், இதை எனக்கு புரியவில்லை.

🎧 Listening... Speak now!
Processing voice...
Assistant: Sorry, I didn’t catch that. Could you repeat?


: 